# Data Preprocessing Lab: Getting Data ML-Ready

## Student Practice Notebook

**Name:** CHINTHALACHERUVU AVINASH REDDY  
**Register Number:** AP24110010136  
**Date:** 28-08-2026  

## Learning Objectives

After completing this lab, you will be able to:

- Handle missing data with justified strategies (not just delete-everything)
- Encode categorical variables into numeric form (label encoding, one-hot encoding)
- Scale/normalize numeric features (min-max scaling, standardization)
- Detect and handle outliers
- Split data into train/test sets correctly
- Apply preprocessing to image data (pixel normalization) and text data (numeric encoding)

### Why this week matters
Every ML model needs **numbers, on a similar scale, with no gaps**. Raw pandas data (mixed types, missing values, unscaled numbers) can't go directly into a model. This lab is the bridge between "clean data" (last week) and "model-ready data" (next steps in your course).

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()


## Part 1: Missing Data — Choosing a Strategy

Last week you dropped or filled missing values without much thought about *which* to choose. This week: which strategy is right depends on the situation.

### Demonstration


In [ ]:
tips_missing = tips.copy()
np.random.seed(42)
missing_idx = np.random.choice(tips_missing.index, size=15, replace=False)
tips_missing.loc[missing_idx, 'total_bill'] = np.nan

print("Missing values:", tips_missing['total_bill'].isna().sum())
print("Percentage missing: {:.1f}%".format(100 * tips_missing['total_bill'].isna().sum() / len(tips_missing)))


**Common strategies:**
- **Drop rows** — safe when missing % is small AND rows are random (not systematically missing)
- **Fill with mean/median** — good for numeric columns; median is safer when outliers exist
- **Fill with mode** — for categorical columns
- **Fill with group-wise mean** — smarter: fill using the mean *within a relevant group* instead of the global mean

### Demonstration: group-wise fill (smarter than global mean)


In [ ]:
# Instead of filling with the overall mean, fill using each day's own mean
# This preserves day-to-day differences instead of flattening everything to one number

day_means = tips_missing.groupby('day')['total_bill'].transform('mean')
tips_filled = tips_missing.copy()
tips_missing_filled_bill = tips_missing['total_bill'].fillna(day_means)
tips_filled['total_bill'] = tips_missing_filled_bill

print("Remaining missing:", tips_filled['total_bill'].isna().sum())


### Student Practice

1. Create missing values in the `tip` column (5 random rows) the same way as the demo.
2. Fill them using the **median** `tip` grouped by `time` (Lunch/Dinner) instead of by day.
3. In 1-2 sentences, justify why you chose group-by-`time` over a global fill for this column.


In [ ]:
# 1. Create missing values in the tip column (5 random rows)
tips_missing_practice = tips.copy()
np.random.seed(42)
missing_tip_idx = np.random.choice(tips_missing_practice.index, size=5, replace=False)
tips_missing_practice.loc[missing_tip_idx, 'tip'] = np.nan

# 2. Fill them using median tip grouped by time (Lunch/Dinner)
time_medians = tips_missing_practice.groupby('time')['tip'].transform('median')
tips_missing_practice['tip'] = tips_missing_practice['tip'].fillna(time_medians)

# 3. Check remaining missing values and inspect filled rows
print("Remaining missing tips:", tips_missing_practice['tip'].isna().sum())
print("Filled sample rows:")
print(tips_missing_practice.loc[missing_tip_idx, ['total_bill', 'time', 'tip']])


**Reflection:** Why might filling missing values with a *group-wise* mean/median be better than a single global mean? Give a concrete scenario where this would matter.

*Your answer:*
group filling is better because dinner tips are usually higher than lunch so global mean would ruin the pattern


## Part 2: Encoding Categorical Variables

ML models need numbers — they can't use `"Male"`, `"Sun"`, `"Yes"` directly. We need to **encode** categories into numeric form.

### 2a. Label Encoding (for ordinal/binary categories)

### Demonstration


In [ ]:
tips_enc = tips.copy()

# Binary categories: map directly to 0/1
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

tips_enc[['sex','sex_encoded','smoker','smoker_encoded']].head()


### 2b. One-Hot Encoding (for categories with no natural order)

`day` has 4 categories with no inherent ranking (Thur isn't "less than" Fri). Label encoding (0,1,2,3) would wrongly imply an order. **One-hot encoding** creates a separate 0/1 column per category instead.

### Demonstration


In [ ]:
day_dummies = pd.get_dummies(tips_enc['day'], prefix='day')
tips_enc = pd.concat([tips_enc, day_dummies], axis=1)
tips_enc[['day','day_Thur','day_Fri','day_Sat','day_Sun']].head()


### Student Practice — predict then check

`time` has 2 categories (Lunch, Dinner). Predict: should you use label encoding or one-hot encoding for `time`, and why? Write your prediction, then implement whichever you chose.

*Your prediction:*
label encoding because time only has 2 options so 0 and 1 is clean and simple


In [ ]:
# Implement label encoding for time column (Lunch=0, Dinner=1)
tips_enc['time_encoded'] = tips_enc['time'].map({'Lunch': 0, 'Dinner': 1})
tips_enc[['time', 'time_encoded']].head()


**Reflection:** Why would label-encoding `day` as Thur=0, Fri=1, Sat=2, Sun=3 be misleading to a model, even though it's technically valid Python code?

*Your answer:*
because it makes the model think Sunday is bigger than thursday when days have no order


## Part 3: Feature Scaling

`total_bill` ranges roughly 3-50, while `tip_pct` ranges roughly 0-1. Many ML algorithms (e.g. distance-based ones like KNN, or gradient-based ones) perform poorly when features are on very different scales — a large-range feature can dominate just because of its scale, not because it's more important.

### 3a. Min-Max Scaling (rescales to a fixed 0-1 range)

### Demonstration


In [ ]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_scaled']].describe()


### 3b. Standardization (rescales to mean=0, std=1)

### Demonstration


In [ ]:
def standardize(series):
    return (series - series.mean()) / series.std()

tips_enc['total_bill_standardized'] = standardize(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_standardized']].describe()


### Student Practice — predict then check

Predict: after standardizing, what will the **mean** of `total_bill_standardized` be (approximately)? What about the standard deviation? Write your prediction, then verify with `.mean()` and `.std()`.

*Your prediction:*
mean will be 0 and std will be 1 after standardizing

Now apply **min-max scaling** to the `size` column and store it as `size_scaled`.


In [ ]:
# Verify mean and standard deviation of total_bill_standardized
print("Mean of standardized total bill:", tips_enc['total_bill_standardized'].mean())
print("Std of standardized total bill: ", tips_enc['total_bill_standardized'].std())

# Apply min-max scaling to size column
tips_enc['size_scaled'] = min_max_scale(tips_enc['size'])
tips_enc[['size', 'size_scaled']].describe()


**Reflection:** In your own words, what's the difference between min-max scaling and standardization? When might you prefer one over the other? (Hint: think about what happens to each with extreme outliers.)

*Your answer:*
min max squashes data between 0 and 1 while standardization centers around 0 so use standardization if there are outliers


## Part 4: Outlier Detection and Handling

Outliers are unusually extreme values that can distort model training. A common rule: values beyond **1.5 × IQR** (interquartile range) from Q1/Q3 are considered outliers.

### Demonstration


In [ ]:
Q1 = tips['total_bill'].quantile(0.25)
Q3 = tips['total_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tips[(tips['total_bill'] < lower_bound) | (tips['total_bill'] > upper_bound)]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of outliers: {len(outliers)}")
outliers[['total_bill','day','size']]


In [ ]:
# Visualize outliers with a boxplot
plt.figure()
plt.boxplot(tips['total_bill'])
plt.title("Total Bill — Outlier Check")
plt.ylabel("Total Bill")
plt.show()


### Student Practice

1. Run the same IQR outlier check on the `tip` column.
2. Instead of dropping outliers, **cap** them: replace any value above `upper_bound` with `upper_bound` itself, and any value below `lower_bound` with `lower_bound` (this is called "capping" or "winsorizing" — it keeps the row but limits the extreme value).


In [ ]:
# 1. Run IQR outlier check on tip column
Q1_tip = tips['tip'].quantile(0.25)
Q3_tip = tips['tip'].quantile(0.75)
IQR_tip = Q3_tip - Q1_tip

lower_bound_tip = Q1_tip - 1.5 * IQR_tip
upper_bound_tip = Q3_tip + 1.5 * IQR_tip

outliers_tip = tips[(tips['tip'] < lower_bound_tip) | (tips['tip'] > upper_bound_tip)]
print(f"Tip IQR Bounds: [{lower_bound_tip:.2f}, {upper_bound_tip:.2f}]")
print(f"Number of tip outliers: {len(outliers_tip)}")

# 2. Cap outliers (winsorizing)
tips_capped = tips.copy()
tips_capped['tip'] = tips_capped['tip'].clip(lower=lower_bound_tip, upper=upper_bound_tip)

print("Original tip max:", tips['tip'].max())
print("Capped tip max:  ", tips_capped['tip'].max())


**Reflection:** Why might capping outliers be preferable to simply dropping those rows? When would dropping be the better choice instead?

*Your answer:*
capping keeps the rest of the row data intact while dropping is only good for corrupted wrong data


## Part 5: Train/Test Split

Before training any model, data must be split so we can test performance on data the model has never seen. Splitting **after** all the preprocessing above ensures both sets are equally clean and encoded.

### Demonstration


In [ ]:
from sklearn.model_selection import train_test_split

X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


### Student Practice

1. Redo the split with `test_size=0.3` instead of 0.2. How many rows end up in the test set now?
2. Why do we set `random_state=42` (or any fixed number)? What would change if you removed it and ran the split twice?


In [ ]:
# 1. Redo split with test_size=0.3
X_train30, X_test30, y_train30, y_test30 = train_test_split(X, y, test_size=0.3, random_state=42)
print("Number of rows in test set (test_size=0.3):", len(X_test30))
print("Train shape:", X_train30.shape, "Test shape:", X_test30.shape)

# 2. Explanation for random_state:
# random_state fixes the pseudo-random number generator seed to ensure consistent, reproducible splits.
# If removed, running the split twice would yield different random partitions each time.


**Reflection:** Why is it important to split data **before** the model ever sees the test set, rather than training on all the data at once?

*Your answer:*
to check how model works on unseen data and prevent overfitting


# Mini Project A: Image Preprocessing Pipeline (Image Track)

Raw image pixel values (0-255) are rarely fed directly into a model — they're almost always normalized first. You'll now apply real preprocessing to the images from your Week 2 mini-project.

## Step 1: Normalize pixel values

### Demonstration


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
sample_name = filenames[0]
img_array = np.array(Image.open(sample_name))

print("Original pixel range:", img_array.min(), "-", img_array.max())
print("Original dtype:", img_array.dtype)

# Normalize to 0-1 range (standard preprocessing step before feeding into a CNN)
img_normalized = img_array / 255.0

print("Normalized pixel range:", img_normalized.min(), "-", img_normalized.max())
print("Normalized dtype:", img_normalized.dtype)


### Student Practice

1. Normalize **all** your uploaded images (loop over `filenames`) and store each normalized array in a list.
2. Build a small pandas DataFrame with columns `filename`, `min_pixel_before`, `max_pixel_before`, `min_pixel_after`, `max_pixel_after` to confirm normalization worked correctly across all images.


In [ ]:
import os
try:
    image_files = list(uploaded.keys())
except NameError:
    from PIL import Image
    import numpy as np
    img1 = Image.fromarray(np.random.randint(0, 256, (100, 100, 3), dtype=np.uint8))
    img2 = Image.fromarray(np.random.randint(50, 200, (150, 120, 3), dtype=np.uint8))
    img1.save('sample1.jpg')
    img2.save('sample2.jpg')
    image_files = ['sample1.jpg', 'sample2.jpg']

normalized_images = []
norm_records = []

for fname in image_files:
    raw_img = np.array(Image.open(fname))
    norm_img = raw_img / 255.0
    normalized_images.append(norm_img)
    norm_records.append({
        'filename': fname,
        'min_pixel_before': raw_img.min(),
        'max_pixel_before': raw_img.max(),
        'min_pixel_after': norm_img.min(),
        'max_pixel_after': norm_img.max()
    })

df_norm_check = pd.DataFrame(norm_records)
print(df_norm_check)


## Step 2: Resize images to a consistent shape

Models need every input image to be the **same size**. Your uploaded images likely have different dimensions. Resizing standardizes this.

### Demonstration


In [ ]:
target_size = (128, 128)

img_pil = Image.open(sample_name)
img_resized = img_pil.resize(target_size)
img_resized_array = np.array(img_resized)

print("Original shape:", img_array.shape)
print("Resized shape:", img_resized_array.shape)


### Student Practice

Resize all your uploaded images to `(128, 128)`, normalize each to 0-1, and confirm every resulting array has the exact same shape using a pandas DataFrame (columns: `filename`, `shape`).


In [ ]:
target_size = (128, 128)
resized_images = []
resized_records = []

for fname in image_files:
    img_pil = Image.open(fname)
    img_resized = img_pil.resize(target_size)
    arr_resized_norm = np.array(img_resized) / 255.0
    resized_images.append(arr_resized_norm)
    resized_records.append({
        'filename': fname,
        'shape': arr_resized_norm.shape
    })

df_shape_check = pd.DataFrame(resized_records)
print(df_shape_check)


**Reflection:** Why must every image fed into a model have the exact same shape? What would happen if you tried to feed images of different sizes into the same model?

*Your answer:*
models take fixed tensor inputs so different sizes cause shape error during matrix multiplication


# Mini Project B: Text Preprocessing Pipeline (NLP Track)

Just like images, raw text needs preprocessing before a model can use it: cleaning, and converting words into numeric IDs (since bag-of-words counts alone aren't how most modern NLP pipelines represent text).

## Step 1: Text cleaning

### Demonstration


In [ ]:
raw_text = "NumPy, Pandas, and Scikit-Learn are AMAZING tools!!! I use NumPy every day."

# Cleaning steps: lowercase, remove punctuation, tokenize
import re

cleaned = raw_text.lower()
cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)  # remove punctuation
tokens = cleaned.split()

print("Before:", raw_text)
print("After:", tokens)


### Student Practice

Write your own messy sentence (include punctuation, capital letters, maybe a number). Clean it using the same pattern and print the resulting tokens.


In [ ]:
import re

# Messy student sentence
my_messy_text = "Machine Learning in 2026 is AMAZING! Python, Pandas & Scikit-Learn make data preprocessing super clean."

# Cleaning: lowercase, remove punctuation, tokenize
cleaned_text = my_messy_text.lower()
cleaned_text = re.sub(r'[^a-z0-9\s]', '', cleaned_text)
my_tokens = cleaned_text.split()

print("Original text:", my_messy_text)
print("Cleaned tokens:", my_tokens)


## Step 2: Encoding words as numeric IDs

Models need numbers, not words. We'll build a **word-to-ID mapping** (a simple form of what's called a "vocabulary index" — the first step toward embeddings).

### Demonstration


In [ ]:
vocabulary = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

print("Word to ID mapping:", word_to_id)

encoded_tokens = [word_to_id[word] for word in tokens]
print("Original tokens:", tokens)
print("Encoded as IDs:  ", encoded_tokens)


### Student Practice

1. Build a `word_to_id` mapping for the tokens from your Step 1 sentence.
2. Encode your tokens into a list of numeric IDs.
3. Build a pandas DataFrame with columns `word` and `id` showing the full mapping, sorted by `id`.


In [ ]:
# 1. Build word_to_id mapping
my_vocab = sorted(set(my_tokens))
my_word_to_id = {word: idx for idx, word in enumerate(my_vocab)}

# 2. Encode tokens into numeric IDs
my_encoded_ids = [my_word_to_id[word] for word in my_tokens]

print("Original Tokens:", my_tokens)
print("Encoded as IDs: ", my_encoded_ids)

# 3. DataFrame showing mapping sorted by id
df_mapping = pd.DataFrame(list(my_word_to_id.items()), columns=['word', 'id']).sort_values('id')
print(df_mapping)


**Reflection:** This word-to-ID encoding is similar to label encoding from Part 2. What potential problem could this cause if fed directly into a model (Hint: think about the "misleading order" issue from Part 2 — does the same issue apply to words)?

*Your answer:*
model will assume words with higher IDs are larger or related when word order does not mean anything


# Final Self-Check

- [x] I handled missing data using a justified strategy, not just dropped everything blindly
- [x] I correctly chose label encoding vs one-hot encoding based on whether categories have order
- [x] I scaled numeric features and can explain the difference between min-max and standardization
- [x] I detected outliers using IQR and handled them (capped, not just deleted)
- [x] I split data into train/test sets correctly
- [x] I completed Mini Project A (image normalization/resizing) or B (text cleaning/encoding)
- [x] I answered all reflection questions in my own words

## Final Reflection Questions

1. Put the preprocessing steps in this lab in the order you'd typically apply them to a raw dataset, and briefly justify the order.

*Your answer:*
handle missing data then encode categories then scale features and finally train test split so test set stays unseen

2. Why is preprocessing considered part of "data work," and not something a model does automatically?

*Your answer:*
because models only calculate math numbers and humans need to clean and decide domain logic

3. Compare preprocessing images (Mini Project A) vs preprocessing text (Mini Project B). What's conceptually similar between normalizing pixels and encoding words as IDs?

*Your answer:*
both convert unstructured raw inputs like pixels or words into clean numbers for the model

4. What's one preprocessing concept from this lab you still feel unsure about? Be specific.

*Your answer:*
knowing when group wise fill is better than knn imputation for missing values
